## GridLock 2.0 - Travel Demand Forecast

Day 48 in `train.csv` is a full day covering the same hours we predict on day 49,
so each location's day-48 demand curve drives the model. On top of that I add
time-of-day, spatial (geohash) context, the road/weather fields and short lags,
then blend LightGBM, XGBoost and CatBoost.

`train.csv` is the only data used to fit the model.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import pygeohash as pgh
from sklearn.cluster import KMeans
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

def add_time(df):
    parts = df["timestamp"].str.split(":", expand=True).astype(int)
    df["hour"], df["minute"] = parts[0], parts[1]
    df["time_bucket"] = df["hour"] * 4 + df["minute"] // 15
    return df

train = add_time(pd.read_csv("train.csv"))
test = add_time(pd.read_csv("test.csv"))
print(train.shape, test.shape)

In [ ]:
# geohash -> coordinates, then cluster nearby cells and list adjacent ones
cells = list(set(train["geohash"]) | set(test["geohash"]))
coord = {g: (pgh.decode(g).latitude, pgh.decode(g).longitude) for g in cells}
km = KMeans(n_clusters=40, random_state=SEED, n_init=10)
cluster = dict(zip(cells, km.fit_predict(np.array([coord[g] for g in cells]))))

steps = ["top", "bottom", "left", "right", "topleft", "topright", "bottomleft", "bottomright"]
known = set(cells)
adj = {}
for g in cells:
    try:
        adj[g] = [pgh.get_adjacent(g, s) for s in steps if pgh.get_adjacent(g, s) in known]
    except Exception:
        adj[g] = []

In [ ]:
# day-48 demand summaries (the history the test day is compared against)
hist = train[train["day"] == 48].copy()
bucket_demand = hist.groupby(["geohash", "time_bucket"])["demand"].mean().to_dict()
hour_demand = hist.groupby(["geohash", "hour"])["demand"].mean().to_dict()
g_mean = hist.groupby("geohash")["demand"].mean().to_dict()
g_std = hist.groupby("geohash")["demand"].std().to_dict()
g_max = hist.groupby("geohash")["demand"].max().to_dict()
g_min = hist.groupby("geohash")["demand"].min().to_dict()
g_med = hist.groupby("geohash")["demand"].median().to_dict()
hourly = hist.groupby("hour")["demand"].mean().to_dict()
bucketly = hist.groupby("time_bucket")["demand"].mean().to_dict()
cluster_demand = hist.assign(c=hist["geohash"].map(cluster)).groupby("c")["demand"].mean().to_dict()
adj_demand = {g: np.mean([g_mean.get(n, 0) for n in ns]) if ns else g_mean.get(g, 0)
              for g, ns in adj.items()}
temp_fill = hist["Temperature"].median()

curve = {g: np.array([bucket_demand.get((g, b), np.nan) for b in range(96)]) for g in cells}

def around(g, b, w, fn):
    seg = curve[g][max(0, b - w):b + w + 1]
    seg = seg[~np.isnan(seg)]
    return fn(seg) if len(seg) else g_mean.get(g, 0)

In [ ]:
def build(df):
    df = df.copy()
    df["lat"] = df["geohash"].map(lambda x: coord.get(x, (0, 0))[0])
    df["lon"] = df["geohash"].map(lambda x: coord.get(x, (0, 0))[1])
    df["cluster"] = df["geohash"].map(cluster).fillna(0).astype(int)
    df["dow"] = (df["day"] - 1) % 7

    df["is_peak"] = (((df["hour"] >= 7) & (df["hour"] <= 9)) |
                     ((df["hour"] >= 17) & (df["hour"] <= 19))).astype(int)
    df["is_night"] = ((df["hour"] >= 22) | (df["hour"] <= 5)).astype(int)
    df["is_lunch"] = ((df["hour"] >= 11) & (df["hour"] <= 13)).astype(int)
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["bkt_sin"] = np.sin(2 * np.pi * df["time_bucket"] / 96)
    df["bkt_cos"] = np.cos(2 * np.pi * df["time_bucket"] / 96)

    df["road"] = df["RoadType"].map({"Residential": 0, "Street": 1, "Highway": 2}).fillna(-1)
    df["large"] = (df["LargeVehicles"] == "Allowed").astype(int)
    df["landmark"] = (df["Landmarks"] == "Yes").astype(int)
    df["weather"] = df["Weather"].map({"Sunny": 0, "Rainy": 1, "Foggy": 2, "Snowy": 3}).fillna(-1)
    df["temp"] = df["Temperature"].fillna(temp_fill)

    # day-48 demand at the same slot and the four neighbouring slots
    for off in (-2, -1, 0, 1, 2):
        df[f"d48_b{off}"] = df.apply(
            lambda r: bucket_demand.get((r["geohash"], r["time_bucket"] + off),
                                        g_mean.get(r["geohash"], 0)), axis=1)
    df["d48_w3m"] = df.apply(lambda r: around(r["geohash"], r["time_bucket"], 3, np.mean), axis=1)
    df["d48_w3s"] = df.apply(lambda r: around(r["geohash"], r["time_bucket"], 3, np.std), axis=1)
    df["d48_hour"] = df.apply(lambda r: hour_demand.get((r["geohash"], r["hour"]),
                                                         g_mean.get(r["geohash"], 0)), axis=1)

    df["geo_mean"] = df["geohash"].map(g_mean).fillna(0)
    df["geo_std"] = df["geohash"].map(g_std).fillna(0)
    df["geo_max"] = df["geohash"].map(g_max).fillna(0)
    df["geo_min"] = df["geohash"].map(g_min).fillna(0)
    df["geo_med"] = df["geohash"].map(g_med).fillna(0)
    df["hour_glob"] = df["hour"].map(hourly).fillna(0)
    df["bkt_glob"] = df["time_bucket"].map(bucketly).fillna(0)
    df["cl_mean"] = df["cluster"].map(cluster_demand).fillna(0)
    df["nbr_mean"] = df["geohash"].map(adj_demand).fillna(0)

    df["temp_weather"] = df["temp"] * df["weather"]
    df["lanes_road"] = df["NumberofLanes"] * df["road"]
    df["d48b0_peak"] = df["d48_b0"] * df["is_peak"]
    df["d48b0_vs_geo"] = df["d48_b0"] / (df["geo_mean"] + 1e-9)
    return df

train = build(train)
test = build(test)

# short lags from each cell's run-up; for test these hold the last observed values
train = train.sort_values(["geohash", "day", "time_bucket"]).reset_index(drop=True)
for i in (1, 2, 3):
    train[f"lag_{i}"] = train.groupby("geohash")["demand"].shift(i)
train["diff_1"] = train["lag_1"] - train["lag_2"]
tail = train.groupby("geohash")["demand"].apply(list)
for i in (1, 2, 3):
    test[f"lag_{i}"] = test["geohash"].map(
        lambda g, n=i: tail.get(g, [0])[-n] if len(tail.get(g, [])) >= n else 0)
test["diff_1"] = test["lag_1"] - test["lag_2"]

In [ ]:
cols = ["lat", "lon", "cluster", "hour", "minute", "time_bucket", "dow",
        "is_peak", "is_night", "is_lunch", "hour_sin", "hour_cos", "bkt_sin", "bkt_cos",
        "road", "NumberofLanes", "large", "landmark", "weather", "temp",
        "d48_b-2", "d48_b-1", "d48_b0", "d48_b1", "d48_b2", "d48_w3m", "d48_w3s", "d48_hour",
        "geo_mean", "geo_std", "geo_max", "geo_min", "geo_med", "hour_glob", "bkt_glob",
        "cl_mean", "nbr_mean", "temp_weather", "lanes_road", "d48b0_peak", "d48b0_vs_geo",
        "lag_1", "lag_2", "lag_3", "diff_1"]

data = train.dropna(subset=["lag_1"]).copy()
X = data[cols].astype(np.float32).replace([np.inf, -np.inf], 0).fillna(0)
y = data["demand"].values
X_test = test[cols].astype(np.float32).replace([np.inf, -np.inf], 0).fillna(0)

# tree count from a day48 -> day49(early) holdout
fit = data[data["day"] == 48]
hold = data[data["day"] == 49]
probe = lgb.LGBMRegressor(objective="regression", metric="rmse", learning_rate=0.03,
                          num_leaves=255, min_child_samples=20, feature_fraction=0.75,
                          bagging_fraction=0.8, bagging_freq=5, reg_alpha=0.1, reg_lambda=0.5,
                          n_estimators=2000, verbose=-1, random_state=SEED, n_jobs=-1)
probe.fit(fit[cols].astype(np.float32).fillna(0), fit["demand"].values,
          eval_set=[(hold[cols].astype(np.float32).fillna(0), hold["demand"].values)],
          callbacks=[lgb.early_stopping(60, verbose=False)])
n_trees = max(120, probe.best_iteration_ or 200)

In [ ]:
# LightGBM averaged over seeds, plus XGBoost and CatBoost
lgb_runs = []
for sd in (42, 7, 2024):
    m = lgb.LGBMRegressor(objective="regression", metric="rmse", learning_rate=0.03,
                          num_leaves=255, min_child_samples=20, feature_fraction=0.75,
                          bagging_fraction=0.8, bagging_freq=5, reg_alpha=0.1, reg_lambda=0.5,
                          n_estimators=n_trees, verbose=-1, random_state=sd, n_jobs=-1)
    m.fit(X, y)
    lgb_runs.append(np.clip(m.predict(X_test), 0, 1))
p_lgb = np.mean(lgb_runs, axis=0)

xgb_m = xgb.XGBRegressor(objective="reg:squarederror", learning_rate=0.03, max_depth=7,
                         min_child_weight=40, subsample=0.8, colsample_bytree=0.75,
                         reg_alpha=0.1, reg_lambda=1.0, n_estimators=n_trees,
                         random_state=SEED, tree_method="hist", n_jobs=-1, verbosity=0)
xgb_m.fit(X, y)
p_xgb = np.clip(xgb_m.predict(X_test), 0, 1)

cat_m = CatBoostRegressor(iterations=n_trees, learning_rate=0.03, depth=7, l2_leaf_reg=3,
                          random_seed=SEED, verbose=0, eval_metric="RMSE", task_type="CPU")
cat_m.fit(X, y)
p_cat = np.clip(cat_m.predict(X_test), 0, 1)

# blend, then widen the spread ~5% (boosted trees regress toward the mean)
blend = 0.6 * p_lgb + 0.25 * p_xgb + 0.15 * p_cat
center = blend.mean()
pred = np.clip(center + 1.05 * (blend - center), 0, 1)

out = pd.DataFrame({"Index": test["Index"].values, "demand": pred})
out.to_csv("submission.csv", index=False)
out.head()